## Setting up

Run the cell below first, before anything else on this page.

Colab hands you an empty machine, so it installs the packages this
tutorial needs, and sets up `login()` — which the download cells use to
get your data provider username and password from your Colab Secrets. It takes a minute or two.

Colab does not keep anything between sessions, so run it again whenever you
reopen this notebook — it is safe to re-run at any point.


In [ ]:
# --- Setup: run this cell first. Safe to re-run. ---------------------------
# Colab gives you an empty machine and takes it back when the session ends,
# so this installs the packages the tutorial needs, and sets up how the
# download cells get your data provider logins. Run it again whenever you
# reopen the notebook.

import importlib
import importlib.util
import os
import subprocess
import sys

try:
    import google.colab               # noqa: F401 - importing it is the test
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED = {                      # import name -> pip name
    "requests": "requests",
    "tqdm": "tqdm",
}

missing = [spec for mod, spec in REQUIRED.items()
           if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing],
                   check=False)
    importlib.invalidate_caches()

    # Check they import, rather than that pip said it was fine: a package can
    # install cleanly and still fail to load, and the fix for that is a
    # restart, which pip cannot do for you.
    broken = []
    for mod in REQUIRED:
        try:
            importlib.import_module(mod)
        except Exception as error:        # noqa: BLE001 - report anything
            broken.append(f"{mod} ({type(error).__name__}: {error})")

    if broken:
        print("\nInstalled, but these still will not import:")
        for line in broken:
            print("   ", line)
        print("Runtime -> Restart session, then run this cell again.")
    else:
        print("Ready.")
else:
    print("Ready - nothing to install.")


# --- Signing in ------------------------------------------------------------
# This defines `login()`. The download cells below call it when they need your
# username and password for a data provider; nothing is asked for here.
#
# It looks in **Colab Secrets** first - the key icon in the left sidebar. A
# secret belongs to your Google account rather than to this notebook, so you
# add it once and every tutorial in the course finds it. Turn on "Notebook
# access" for each. This tutorial asks for:
#
#     CDSE_USERNAME  and  CDSE_PASSWORD
#         the email address you registered with, and its password
#
# There is no typing them in here instead. A notebook can read Secrets but
# cannot write them, so a password typed into a cell would be gone again next
# session - and saved into the notebook if you shared it. `login()` says what
# to add and where if a secret is missing.

CREDENTIALS = {            # what to call it -> secret names, and sign-up
    "Copernicus Data Space":
        ("CDSE_USERNAME", "CDSE_PASSWORD",
         "https://dataspace.copernicus.eu/"),
}


def read_secret(name):
    """One Colab secret, or None if unset or not shared with this notebook."""
    if not IN_COLAB:
        return None
    try:
        from google.colab import userdata

        return (userdata.get(name) or "").strip() or None
    except Exception:                 # noqa: BLE001 - unset, or not shared
        return None


def login(provider):
    """The username and password for one data provider, from Colab Secrets.

    Nothing is typed into this notebook. Secrets are re-read on every call, so
    correcting one in the sidebar and running the cell again takes effect.
    """
    user_secret, pass_secret, register = CREDENTIALS[provider]
    user, password = read_secret(user_secret), read_secret(pass_secret)

    if not (user and password):
        raise RuntimeError(
            f"{provider}: {user_secret} and {pass_secret} are not readable.\n\n"
            "Open the key icon in the left sidebar - the Secrets tab - and for\n"
            "each of the two:\n"
            "    1. click + Add new secret\n"
            f"    2. put {user_secret} (then {pass_secret}) in Name\n"
            "    3. paste the value in Value\n"
            "    4. switch Notebook access on\n\n"
            "Then run this cell again. If they are already there, it is the\n"
            "Notebook access switch that is off for this notebook.\n\n"
            f"No account yet? Register at {register}")

    return user, password


> **Colab: this notebook and the analysis notebook do not share a machine.**
> The scene you download here is not there when you open
> `california_wildfires_colab.ipynb` — on Colab every notebook gets its own
> runtime, and nothing on disk carries between them.
>
> So do not split the work. Copy the cell below into the analysis notebook and
> run it there, once per scene, so the `.SAFE` folders land in the same session
> that reads them.


In [ ]:
import os
import zipfile
import io
import requests
from tqdm import tqdm

# Change PRODUCT_NAME to the SAFE file you want to download.
PRODUCT_NAME = "S2A_MSIL1C_20250102T183751_N0511_R027_T11SLT_20250102T202910.SAFE"

# Save to the folder data/.
output_dir = os.path.join("data", PRODUCT_NAME)
os.makedirs("data", exist_ok=True)

# Check if the .SAFE directory already exists and is non-empty
if os.path.exists(output_dir) and os.listdir(output_dir):
    print(f"Skipping download: '{PRODUCT_NAME}' is already downloaded in data/.")
else:
    # 1. Fetch Product ID
    search_url = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Name eq '{PRODUCT_NAME}'"
    response = requests.get(search_url).json()
    if not response.get("value"):
        raise ValueError(f"Product '{PRODUCT_NAME}' was not found.")
    product_id = response["value"][0]["Id"]
    download_url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products({product_id})/$value"

    # 2. Authenticate
    username, password = login("Copernicus Data Space")
    token = requests.post(
        "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
        data={
            "client_id": "cdse-public",
            "username": username,
            "password": password,
            "grant_type": "password",
        },
    ).json()["access_token"]

    # 3. Stream and extract directly into the SAFE folder
    with requests.get(
        download_url, headers={"Authorization": f"Bearer {token}"}, stream=True
    ) as r:
        r.raise_for_status()
        total_size = int(r.headers.get("content-length", 0))
        chunk_size = 8192
        with (
            io.BytesIO() as buffer,
            tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                unit_divisor=1024,
                desc="Downloading",
            ) as bar,
        ):
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    buffer.write(chunk)
                    bar.update(len(chunk))
            buffer.seek(0)
            with zipfile.ZipFile(buffer) as zf:
                # Extracts into data/ folder, using the root path inside the zip file
                zf.extractall("data")

    print(f"Saved to {output_dir}")